# Poutre sans charge transverse : branches en fonction de k (C1)

Lit `results/c1/branches.csv`, `critical.csv` et `neurom_points.csv`, produits par `branches_k.py`.
Plan : `Plan-tests-cas-difficile-non-lineaire.md` (notes).

- μ = k^{1/4} ; (K + kM) φ_n = P_n(k) G φ_n ; branche n : w = ±c_n φ_n, c_n² φ_nᵀGφ_n = 2(P − P_n), existe pour P_n < P.
- Traits fins : branches suivies par continuité en k ; trait épais : minimum global (plus petit P_n, seule branche stable).
- Points : NeuROM (LBFGS préconditionné) ; ○ meilleur des départs 0,1·φ_n, × départ bruit.

Figures des notes d'expérience : chaque figure insérée dans une note est produite par une cellule de ce notebook, qui l'enregistre dans `vault/Images/` sous le nom cité dans la note (`save`).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

sys.path.insert(0, "../..")
from plotting import plot_branches_k

branches = pd.read_csv("results/c1/branches.csv")
points = pd.read_csv("results/c1/neurom_points.csv")
critical = pd.read_csv("results/c1/critical.csv").set_index("P")

IMG = Path.home() / "Library/CloudStorage/GoogleDrive-solal21a@gmail.com/Mon Drive/Solal_PhD/vault/Images"


def save(fig, name, dpi=130):
    """Save a figure of an experiment note into the vault images (skipped if the vault is absent)."""
    if IMG.exists():
        fig.savefig(IMG / name, dpi=dpi)
points.groupby(["P", "start"]).agg(sur_branche_1=("branch", lambda s: (s == 1).mean()),
                                     lambda_min=("lambda_min", "min"), err_max=("err_branch", "max"))

## Diagrammes w(1/4) – μ

In [ ]:
for P in sorted(branches.P.unique()):
    plot_branches_k(branches, points, P, mu_critical=critical.mu_cr[P]).show()

## POD (x, μ) du minimum global, par niveau de P

Expérience : [Exp-winkler-calibration](obsidian://open?vault=vault&file=5_Projects%2F1A%2FNLPGD_test_bifurcation%2FExp%C3%A9riences%2FExp-winkler-calibration), section « C1, étape 2 ».

Lit `results/c1/pod.csv` (`pod.py --study c1`) et `results/c1/snapshots/` (`snapshots_k.py`).

- Snapshots NeuROM : 150 valeurs de μ sur [0, μ_cr(P)), continuation en k : départ = solution au μ précédent + bruit 1e-4, relance depuis un col avec 1e-2 (`Exp-winkler-continuation.md`, variante (c)).
- σ_n, ε(n), n(ε) : SVD pondérée (norme L² exacte en x, trapèzes en μ). `_exact` : mêmes calculs sur c₁φ₁(k).
- P : un P au milieu de chaque intervalle entre les charges des sauts (70 à 2200, 0 à 9 sauts).
- `_exact` désigne la référence discrète par valeurs propres (même maillage), pas une forme fermée.
- n_jumps : nombre de sauts du minimum global (changements de symétrie de φ₁) sur [0, μ_cr).

In [ ]:
pod_c1 = pd.read_csv("results/c1/pod.csv")
pod_c1[[c for c in pod_c1.columns if not c.startswith("sigma_")]]

### Décroissance de la POD

Figure de la note : `Exp-winkler-calibration-C1-pod.png`.

In [ ]:
# Note : Exp-winkler-calibration, C1 étape 2 -> Exp-winkler-calibration-C1-pod.png
sigma_cols = sorted([c for c in pod_c1.columns if c.startswith("sigma_") and c[6:].isdigit()], key=lambda c: int(c[6:]))
fig, axs = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
for i, r in pod_c1.iterrows():
    s = r[sigma_cols].to_numpy(float)
    s = s[~np.isnan(s)]
    tail = np.sqrt(np.cumsum((s**2)[::-1])[::-1] / (s**2).sum())   # eps(n - 1), n = 1..
    eps = np.append(tail[1:], 0.0)[:30]
    axs[0].semilogy(np.arange(1, len(eps) + 1), np.maximum(eps, 1e-16), color=plt.cm.viridis(i / (len(pod_c1) - 1)),
                    label=f"P = {r.P:g} ({int(r.n_jumps)})")
axs[0].set(xlabel="n (nombre de modes)", ylabel="ε(n)", title="Erreur relative de la POD de rang n", ylim=(1e-14, 2))
axs[0].legend(fontsize=7, title="P (sauts)")
axs[1].plot(pod_c1.n_jumps, pod_c1["n_1e-02"], "o-", label="n(10⁻²)")
axs[1].plot(pod_c1.n_jumps, pod_c1["n_1e-03"], "s-", label="n(10⁻³)")
axs[1].plot(pod_c1.n_jumps, pod_c1["n_1e-04"], "^-", label="n(10⁻⁴)")
axs[1].set(xlabel="nombre de sauts du minimum global", ylabel="nombre de modes", title="Modes nécessaires")
axs[1].legend()
axs[1].grid(alpha=0.3)
save(fig, "Exp-winkler-calibration-C1-pod.png")
plt.show()

### Cols restants après relance

Snapshots où λ_min ≤ 0 (col) malgré la relance ; `saut_avant` : écart en μ au saut du minimum global qui précède.

In [ ]:
runs_c1 = pd.read_csv("results/c1/runs.csv")


def jumps_of(g):
    """Values of mu of the jumps of the global minimum (changes of symmetry of phi_1)."""
    g = g.reset_index(drop=True)
    return [0.5 * (g.mu[j] + g.mu[j + 1]) for j in range(len(g) - 1) if g.symmetric[j] != g.symmetric[j + 1]]


jumps_c1 = {P: jumps_of(g) for P, g in runs_c1.groupby("P")}
saddles = runs_c1[runs_c1.lambda_min <= 0][["P", "mu", "retried", "lambda_min", "err_exact"]].copy()
saddles["saut_avant"] = [mu - max(m for m in jumps_c1[P] if m < mu) for P, mu in zip(saddles.P, saddles.mu)]
saddles

### Cartes w(x, μ)

Snapshots NeuROM pour 0, 4 et 9 sauts. Pointillés : sauts du minimum global ; croix : cols restants (λ_min ≤ 0). Figure de la note : `Exp-winkler-calibration-C1-cartes.png`.

In [ ]:
# Note : Exp-winkler-calibration, C1 étape 2 -> Exp-winkler-calibration-C1-cartes.png
fig, axs = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)
for ax, P in zip(axs, [70.0, 620.0, 2200.0]):
    data = torch.load(f"results/c1/snapshots/P{P:g}.pt", weights_only=False)
    W = data["W"].numpy()
    vmax = np.abs(W).max()
    im = ax.pcolormesh(data["mu"].numpy(), data["x"].numpy(), W, cmap="RdBu_r", vmin=-vmax, vmax=vmax, shading="nearest")
    for m in jumps_c1[P]:
        ax.axvline(m, color="k", ls=":", lw=1)
    for m in saddles[saddles.P == P].mu:
        ax.plot(m, 0.5, "kx", ms=10, mew=2)
    ax.set(title=f"P = {P:g} ({data['n_jumps']} sauts)", xlabel="μ = k^(1/4)", ylabel="x")
    fig.colorbar(im, ax=ax, label="w")
save(fig, "Exp-winkler-calibration-C1-cartes.png")
plt.show()

### Déformée et diagramme de bifurcation le long de μ, P = 2200

Gauche : snapshot NeuROM w(x) au μ courant. Droite : diagramme w(1/4)–μ, branches exactes (traits fins), minimum global (trait épais), μ_cr (tirets) ; les points NeuROM s'ajoutent jusqu'au μ courant (rouge : col). Curseur : 150 valeurs de μ ; ▶ Play fait défiler μ (80 ms par image), ❚❚ Pause l'arrête. Titre : symétrie de φ₁, sauts franchis, λ_min.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from branches_k import PROBE

P = 2200.0
data = torch.load(f"results/c1/snapshots/P{P:g}.pt", weights_only=False)
x, mu, W = data["x"].numpy(), data["mu"].numpy(), data["W"].numpy()
w_quarter = data["Q"][PROBE].numpy()                             # w(1/4) of the snapshots
g = runs_c1[runs_c1.P == P].reset_index(drop=True)
is_saddle = (g.lambda_min <= 0).to_numpy()
b = branches[branches.P == P]


def caption(j):
    n_before = sum(m < mu[j] for m in jumps_c1[P])
    state = "col" if is_saddle[j] else "minimum"
    return (f"P = {P:g}, μ = {mu[j]:.2f} (k = {mu[j]**4:.3g}) : {'symétrique' if g.symmetric[j] else 'antisymétrique'}, "
            f"{n_before} saut(s) franchi(s), λ_min = {g.lambda_min[j]:.2e} ({state})")


def path(j):
    """NeuROM points up to mu_j (blue: minimum, red: saddle) and the current point."""
    colors = np.where(is_saddle[:j + 1], "red", "royalblue")
    return [go.Scatter(x=x, y=W[:, j]),
            go.Scatter(x=mu[:j + 1], y=w_quarter[:j + 1], marker=dict(color=colors)),
            go.Scatter(x=[mu[j]], y=[w_quarter[j]])]


fig = make_subplots(rows=1, cols=2, column_widths=[0.45, 0.55], horizontal_spacing=0.08)
first = path(0)
fig.add_trace(first[0].update(mode="lines", line=dict(color="black"), showlegend=False), row=1, col=1)
fig.add_trace(first[1].update(mode="markers", marker_size=5, name="NeuROM"), row=1, col=2)
fig.add_trace(first[2].update(mode="markers", marker=dict(size=13, color="black", symbol="circle-open", line_width=2),
                              name="μ courant"), row=1, col=2)
moving = list(range(3))                                          # traces updated by the frames
for n, d in b.groupby("branch"):                                 # exact branches, fixed background
    d = d.sort_values("mu")
    for s in (1, -1):
        fig.add_trace(go.Scatter(x=d.mu, y=s * d.w_quarter, mode="lines", line=dict(width=0.7), opacity=0.6,
                                 showlegend=False, hoverinfo="skip"), row=1, col=2)
gm = b[b["global"]].sort_values("mu")
y = gm.w_quarter.to_numpy().copy()
y[np.r_[False, np.diff(gm.branch) != 0]] = np.nan                # break the line at the jumps
for s in (1, -1):
    fig.add_trace(go.Scatter(x=gm.mu, y=s * y, mode="lines", line=dict(color="black", width=3),
                             name="minimum global (exact)", showlegend=s == 1, hoverinfo="skip"), row=1, col=2)
fig.add_vline(x=critical.mu_cr[P], line_dash="dash", line_color="gray", row=1, col=2)

frames = [go.Frame(name=f"{mu[j]:.2f}", data=path(j), traces=moving, layout=dict(title_text=caption(j)))
          for j in range(len(mu))]
fig.frames = frames
play = dict(frame=dict(duration=80, redraw=True), transition=dict(duration=0), fromcurrent=True, mode="immediate")
pause = dict(frame=dict(duration=0, redraw=False), transition=dict(duration=0), mode="immediate")
steps = [dict(method="animate", label=f.name, args=[[f.name], pause]) for f in frames]
ymax = 1.05 * np.abs(W).max()
fig.update_xaxes(title_text="x", row=1, col=1)
fig.update_yaxes(title_text="w(x)", range=[-ymax, ymax], row=1, col=1)
fig.update_xaxes(title_text="μ = k^(1/4)", range=[-0.3, 1.03 * critical.mu_cr[P]], row=1, col=2)
fig.update_yaxes(title_text="w(1/4)", row=1, col=2)
fig.update_layout(title=caption(0), template="plotly_white", height=600, legend=dict(x=1.0, y=1.0, xanchor="right"),
                  updatemenus=[dict(type="buttons", direction="left", x=0, y=-0.12, xanchor="left", yanchor="top",
                                    buttons=[dict(label="▶ Play", method="animate", args=[None, play]),
                                             dict(label="❚❚ Pause", method="animate", args=[[None], pause])])],
                  sliders=[dict(active=0, steps=steps, currentvalue=dict(prefix="μ = "), x=0.2, len=0.8, y=-0.05,
                                pad=dict(t=40))])
fig

### Diagramme de bifurcation, P = 2200 (statique)

Branches exactes (traits fins), minimum global (trait épais), μ_cr (tirets) ; les 150 snapshots NeuROM (points bleus ; croix rouges : cols restants, λ_min ≤ 0).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.5), constrained_layout=True)
for n, d in b.groupby("branch"):
    d = d.sort_values("mu")
    for s in (1, -1):
        ax.plot(d.mu, s * d.w_quarter, color=f"C{n % 10}", lw=0.7)
for s in (1, -1):
    ax.plot(gm.mu, s * y, "k", lw=2.5, label="minimum global (exact)" if s == 1 else None)
ax.plot(mu[~is_saddle], w_quarter[~is_saddle], "o", ms=3.5, color="royalblue", label="NeuROM (minimum)")
ax.plot(mu[is_saddle], w_quarter[is_saddle], "x", ms=10, mew=2, color="red", label="NeuROM : col (λ_min ≤ 0)")
ax.axvline(critical.mu_cr[P], color="gray", ls="--", lw=1)
ax.axhline(0, color="gray", lw=0.5)
ax.set(xlim=(-0.3, 1.03 * critical.mu_cr[P]), xlabel="μ = k^(1/4)   (tirets : μ_cr)", ylabel="w(1/4)",
       title=f"P = {P:g} : branches exactes et 150 snapshots NeuROM ({len(jumps_c1[P])} sauts, {int(is_saddle.sum())} col(s))")
ax.legend(fontsize=8, loc="upper right")
plt.show()

### Animation GIF, P = 2200

Même contenu que l'animation plotly (déformée | diagramme construit au fil de μ), enregistrée dans `vault/Images/Exp-winkler-calibration-C1-P2200-animation.gif` (12 images/s, 150 images).

In [ ]:
# Vault : Exp-winkler-calibration-C1-P2200-animation.gif
from matplotlib.animation import FuncAnimation, PillowWriter

fig, (ax_w, ax_b) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw=dict(width_ratios=[0.45, 0.55]),
                                 constrained_layout=True)
line_w, = ax_w.plot(x, W[:, 0], "k", lw=1.5)
ax_w.set(xlim=(0, 1), ylim=(-ymax, ymax), xlabel="x", ylabel="w(x)")
ax_w.axhline(0, color="gray", lw=0.5)
for n, d in b.groupby("branch"):
    d = d.sort_values("mu")
    for s in (1, -1):
        ax_b.plot(d.mu, s * d.w_quarter, color=f"C{n % 10}", lw=0.6, alpha=0.6)
for s in (1, -1):
    ax_b.plot(gm.mu, s * y, "k", lw=2.5, label="minimum global (exact)" if s == 1 else None)
ax_b.axvline(critical.mu_cr[P], color="gray", ls="--", lw=1)
ax_b.set(xlim=(-0.3, 1.03 * critical.mu_cr[P]), xlabel="μ = k^(1/4)   (tirets : μ_cr)", ylabel="w(1/4)")
done = ax_b.scatter([], [], s=12, label="NeuROM")
current, = ax_b.plot([], [], "o", ms=11, mfc="none", mec="k", mew=2)
ax_b.legend(fontsize=8, loc="upper right")
title = fig.suptitle("")


def draw(j):
    line_w.set_ydata(W[:, j])
    done.set_offsets(np.c_[mu[:j + 1], w_quarter[:j + 1]])
    done.set_color(np.where(is_saddle[:j + 1], "red", "royalblue"))
    current.set_data([mu[j]], [w_quarter[j]])
    title.set_text(caption(j))
    return line_w, done, current, title


animation = FuncAnimation(fig, draw, frames=len(mu), blit=False)
if IMG.exists():
    animation.save(IMG / "Exp-winkler-calibration-C1-P2200-animation.gif", writer=PillowWriter(fps=12), dpi=70)
plt.close(fig)

## Côté de la branche le long de k, P = 500

Expérience : [Exp-winkler-continuation](obsidian://open?vault=vault&file=5_Projects%2F1A%2FNLPGD_test_bifurcation%2FExp%C3%A9riences%2FExp-winkler-continuation) (vault, `5_Projects/1A/NLPGD_test_bifurcation/Expériences/`).

Lit `results/fondation/side_P500.*` (`branch_side_study_fondation.py`). Trois départs NeuROM, 150 valeurs de μ = k^{1/4} sur [0, μ_cr) :
- (a) `same noise` : q₀ = εξ, même bruit pour tous les μ (ε = 1e-4) ;
- (b) `continuation` : q₀ = q(μ précédent) + εξ ;
- (c) `continuation + retry` : comme (b), et si λ_min ≤ 0, seconde résolution depuis q + 1e-2 ξ.

Déformées : 15 sur 150, μ croissant vers le haut ; pointillés : sauts du minimum global. Titre de chaque panneau : nombre de changements de côté dans un segment et de cols (sur les 150 μ). Figures de la note : `Exp-winkler-continuation-T1.png`, `-relance.png`, `-bifurcation.png`.

In [ ]:
side = torch.load("results/fondation/side_P500.pt", weights_only=False)
side_runs = pd.read_csv("results/fondation/side_P500.csv")
VARIANT_LABELS = {"same noise": "(a) même bruit", "continuation": "(b) continuation + bruit",
                  "continuation + retry": "(c) continuation + bruit, relance si col"}


def deflection_stack(variants, name):
    """15 deflections of 150 per variant, shifted upwards as mu grows; dotted lines: jumps."""
    x, mu = side["x"].numpy(), side["mu"].numpy()
    shown = np.unique(np.linspace(0, len(mu) - 1, 15).round().astype(int))
    step = 2.2 * max(np.abs(W.numpy()[:, shown]).max() for W in side["W"].values())
    fig, axs = plt.subplots(1, len(variants), figsize=(6 * len(variants), 11), sharey=True)
    for ax, v in zip(np.atleast_1d(axs), variants):
        W, rv = side["W"][v].numpy(), side_runs[side_runs.variant == v]
        for i, j in enumerate(shown):
            ax.plot(x, W[:, j] + i * step, "k", lw=1.3)
            ax.text(1.01, i * step, f"μ = {mu[j]:.2f}", va="center", fontsize=8)
        for m in side["jumps"]:
            ax.axhline(np.interp(m, mu[shown], np.arange(len(shown))) * step, color="gray", ls=":", lw=1)
        ax.set_title(f"{VARIANT_LABELS[v]}\n{int(rv.side_change.sum())} changement(s) de côté, "
                     f"{int((rv.lambda_min <= 0).sum())} col(s) sur 150 μ", fontsize=10)
        ax.set_xlabel("x")
        ax.set_yticks([])
    np.atleast_1d(axs)[0].set_ylabel("w(x), décalé vers le haut quand μ = k^(1/4) augmente")
    fig.suptitle("P = 500 — 15 déformées sur 150 ; pointillés : sauts du minimum global")
    fig.tight_layout()
    save(fig, name, dpi=100)
    plt.show()


side_runs.groupby("variant").agg(changements_de_cote=("side_change", "sum"),
                                  cols=("lambda_min", lambda s: int((s <= 0).sum())),
                                  relances=("retried", "sum"), err_max=("err_exact", "max"))

### T1 : même bruit (a) contre continuation (b)

In [ ]:
# Note : Exp-winkler-continuation -> Exp-winkler-continuation-T1.png
deflection_stack(["same noise", "continuation"], "Exp-winkler-continuation-T1.png")

### Relance : continuation (b) contre continuation + relance (c)

In [ ]:
# Note : Exp-winkler-continuation -> Exp-winkler-continuation-relance.png
deflection_stack(["continuation", "continuation + retry"], "Exp-winkler-continuation-relance.png")

### Diagrammes de bifurcation associés

w(1/4) en fonction de μ : branches exactes (traits fins), minimum global (trait épais), solutions NeuROM des 150 μ (points bleus ; croix rouges : cols, λ_min ≤ 0). Un changement de côté = un point qui passe sur la moitié opposée du trait épais.

In [ ]:
# Note : Exp-winkler-continuation -> Exp-winkler-continuation-bifurcation.png
P = 500.0
b = branches[branches.P == P]
fig, axs = plt.subplots(3, 1, figsize=(11, 13), sharex=True)
for ax, (v, label) in zip(axs, VARIANT_LABELS.items()):
    for n, d in b.groupby("branch"):
        d = d.sort_values("mu")
        for s in (1, -1):
            ax.plot(d.mu, s * d.w_quarter, color=f"C{n % 10}", lw=0.7)
    g = b[b["global"]].sort_values("mu")
    y = g.w_quarter.to_numpy().copy()
    y[np.r_[False, np.diff(g.branch) != 0]] = np.nan                # break the line at the jumps
    for s in (1, -1):
        ax.plot(g.mu, s * y, "k", lw=2.5, label="minimum global (exact)" if s == 1 else None)
    rv = side_runs[side_runs.variant == v]
    ok, col = rv[rv.lambda_min > 0], rv[rv.lambda_min <= 0]
    ax.plot(ok.mu, ok.w_quarter, "o", ms=3.5, color="royalblue", label="NeuROM (minimum)")
    if len(col):
        ax.plot(col.mu, col.w_quarter, "x", ms=10, mew=2, color="red", label="NeuROM : col (λ_min ≤ 0)")
    ax.axvline(critical.mu_cr[P], color="gray", ls="--", lw=1)
    ax.axhline(0, color="gray", lw=0.5)
    ax.set_xlim(-0.3, critical.mu_cr[P] * 1.05)
    ax.set_title(f"{label} : {int(rv.side_change.sum())} changement(s) de côté, {len(col)} col(s)", fontsize=10)
    ax.set_ylabel("w(1/4)")
    ax.legend(fontsize=8, loc="upper right")
axs[-1].set_xlabel("μ = k^(1/4)   (tirets : μ_cr)")
fig.suptitle("P = 500 — diagrammes de bifurcation : branches exactes (traits fins) et 150 solutions NeuROM")
fig.tight_layout()
save(fig, "Exp-winkler-continuation-bifurcation.png", dpi=90)
plt.show()

## Imperfection : P = 2200, charge transverse en x = 0,3 (curiosité)

Lit `results/fondation/imperfection_P2200_a0.3_w10.csv` et `.pt` (`imperfection_fondation.py`). Pas de note d'expérience.

- Charge F b_s(x − 0,3) (`bspline_load`, s = 0,02), F = 1922 : flèche linéaire 10 à k = 0, P = 0, du même ordre que la flèche flambée (≈ 15 à 30). Hors du milieu, elle agit sur les modes symétriques et antisymétriques.
- 150 valeurs de μ = k^{1/4} sur [0 ; 1,1 μ_cr] (avec la charge, w ≠ 0 au-delà de μ_cr).
- **μ parcouru dans les deux sens** : d'abord croissant (départ 10⁻⁴ ξ au premier μ), puis décroissant, en partant de la dernière solution du parcours croissant. Chaque résolution suit la variante (c) : départ = solution au μ précédent + 10⁻⁴ ξ, relance depuis un col avec 10⁻² ξ. Un écart entre les deux sens = hystérésis.
- Tracé : branches de la poutre parfaite (traits fins) et son minimum global (trait épais), solutions avec la charge (bleu : μ croissant ; orange : μ décroissant ; croix : cols, λ_min ≤ 0).

In [ ]:
P = 2200.0
b = branches[branches.P == P]


def plot_imperfection(imp):
    """w(1/4) - mu with the transverse load, both sweeps, over the branches of the perfect beam."""
    fig, ax = plt.subplots(figsize=(12, 5.5))
    for n, d in b.groupby("branch"):
        d = d.sort_values("mu")
        for s in (1, -1):
            ax.plot(d.mu, s * d.w_quarter, color="0.75", lw=0.6)
    g = b[b["global"]].sort_values("mu")
    y = g.w_quarter.to_numpy().copy()
    y[np.r_[False, np.diff(g.branch) != 0]] = np.nan                    # break the line at the jumps
    for s in (1, -1):
        ax.plot(g.mu, s * y, "k", lw=2, label="poutre parfaite : minimum global" if s == 1 else None)
    for (direction, label, color, marker) in [("increasing", "imperfection, μ croissant", "royalblue", "o"),
                                              ("decreasing", "imperfection, μ décroissant", "darkorange", ".")]:
        d = imp[imp.direction == direction].sort_values("mu")
        ax.plot(d.mu, d.w_quarter, marker, ms=4, color=color, label=label)
        col = d[d.lambda_min <= 0]
        if len(col):
            ax.plot(col.mu, col.w_quarter, "x", ms=10, mew=2, color="red")
    ax.axvline(critical.mu_cr[P], color="gray", ls="--", lw=1)
    ax.axhline(0, color="gray", lw=0.5)
    ax.set(xlabel="μ = k^(1/4)   (tirets : μ_cr de la poutre parfaite)", ylabel="w(1/4)",
           title=f"P = 2200, charge transverse F = {imp.F[0]:.2g} en x = {imp.a[0]:g} (flèche linéaire {imp.w_imperfection[0]:g})")
    ax.legend(fontsize=8, loc="upper right")
    plt.show()

    both = imp.pivot(index="mu", columns="direction", values="w_quarter")
    print(f"max |w(1/4) croissant - décroissant| = {(both.increasing - both.decreasing).abs().max():.2e}, "
          f"cols {int((imp.lambda_min <= 0).sum())}, relances {int(imp.retried.sum())}")


IMPERFECTION = "results/fondation/imperfection_P2200_a0.3_w10"
imp = pd.read_csv(f"{IMPERFECTION}.csv")
plot_imperfection(imp)

### Déformées w(x) le long de μ

Curseur : valeur de μ. Bleu : parcours μ croissant ; orange (pointillés) : μ décroissant. Les deux courbes se superposent sauf là où les deux sens tombent sur des minima différents (vers μ ≈ 23).

In [ ]:
import plotly.graph_objects as go

imp_w = torch.load(f"{IMPERFECTION}.pt", weights_only=False)
x_imp, mu_imp = imp_w["x"].numpy(), imp_w["mu"].numpy()
W_up, W_down = imp_w["W"]["increasing"].numpy(), imp_w["W"]["decreasing"].numpy()
w_max = 1.05 * max(np.abs(W_up).max(), np.abs(W_down).max())

fig = go.Figure([go.Scatter(x=x_imp, y=W_up[:, 0], name="μ croissant", line=dict(color="royalblue")),
                 go.Scatter(x=x_imp, y=W_down[:, 0], name="μ décroissant", line=dict(color="darkorange", dash="dot"))])
fig.frames = [go.Frame(data=[go.Scatter(y=W_up[:, j]), go.Scatter(y=W_down[:, j])], name=str(j)) for j in range(len(mu_imp))]
fig.update_layout(
    title=f"P = 2200, charge F = {imp_w['F']:.4g} en x = {imp_w['a']:g} : déformée w(x)",
    xaxis_title="x", yaxis=dict(title="w", range=[-w_max, w_max]), template="plotly_white",
    sliders=[dict(currentvalue=dict(prefix="μ = "), steps=[
        dict(method="animate", label=f"{m:.2f}", args=[[str(j)], dict(mode="immediate", frame=dict(duration=0, redraw=False))])
        for j, m in enumerate(mu_imp)])],
)
fig